# Browser-Use SDK를 활용한 Browser tool Live View

## 개요

이 튜토리얼에서는 Browser-Use로 Amazon Bedrock AgentCore Browser tool과 상호 작용하고 브라우저 화면을 실시간으로 확인하는 방법을 알아봅니다.


### 튜토리얼 세부 정보


| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                           |
| Agent 유형          | 단일                                                                             |
| Agentic Framework   | Browser-Use                                                                      |
| LLM 모델            | Anthropic Claude 3.7 Sonnet                                                      |
| 튜토리얼 구성 요소  | Browser-Use로 Browser tool과 상호 작용하고 화면을 실시간으로 확인                |
| 튜토리얼 분야       | 범용                                                                             |
| 예제 난이도         | 쉬움                                                                             |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK, Browser-Use                                  |

### 튜토리얼 아키텍처

이 튜토리얼에서는 Browser-Use와 Browser tool을 함께 사용하면서 브라우저 화면을 실시간으로 확인하는 방법을 설명합니다.  

예제에서는 Browser-Use Agent에 자연어 지시를 보내 Bedrock AgentCore Browser에서 작업을 수행하고, 그 과정을 실시간으로 확인합니다.

<div style="text-align:left">
    <img src="images/browser-tool.png" width="50%"/>
</div>

### 튜토리얼 주요 기능

* Browser tool을 사용하고 화면을 실시간으로 확인
* Browser-Use와 Browser tool을 함께 사용

## 사전 요구 사항

### 이 튜토리얼을 실행하려면 다음이 필요합니다.
* Python 3.11+
* AWS 자격 증명. IAM 역할/사용자에 다음 권한이 있어야 합니다. https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html#browser-credentials-config
* Amazon Bedrock AgentCore SDK
* Browser-Use SDK 

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

### browser-use 문제를 해결하려면 아래 패치 스크립트를 실행하세요.

In [ ]:
%%writefile patch_browser_use.py

#!/usr/bin/env python3
"""browser_use의 session.py를 자동으로 찾아 패치합니다."""

import os
import shutil
import sys
from pathlib import Path

def find_browser_use_path():
    """browser_use 설치 경로를 자동으로 찾습니다."""
    try:
        import browser_use
        browser_use_path = Path(browser_use.__file__).parent
        session_file = browser_use_path / "browser" / "session.py"
        return str(session_file)
    except ImportError:
        print("❌ browser_use not installed. Install with: pip install browser-use")
        return None

def patch_browser_use():
    # 파일 경로 자동 감지
    file_path = find_browser_use_path()
    if not file_path:
        return False
    
    if not os.path.exists(file_path):
        print(f"❌ File not found: {file_path}")
        return False
    
    print(f"📁 Found browser_use at: {file_path}")
    
    # 백업 생성
    backup_path = file_path + ".backup"
    if not os.path.exists(backup_path):
        shutil.copy2(file_path, backup_path)
        print(f"💾 Created backup: {backup_path}")
    else:
        print(f"📋 Backup already exists: {backup_path}")
    
    # 파일 읽기
    with open(file_path, 'r') as f:
        content = f.read()
    
    # 변경 1: cdp_url 확인 후 header 확인 추가
    old1 = "if not cdp_url:\n\t\t\tprofile_kwargs['is_local'] = True"
    new1 = "if not cdp_url:\n\t\t\tprofile_kwargs['is_local'] = True\n\n\t\tif headers:\n\t\t\tprofile_kwargs['headers'] = headers"
    
    if old1 in content and "if headers:\n\t\t\tprofile_kwargs['headers'] = headers" not in content:
        content = content.replace(old1, new1)
        print("✅ Added headers check")
    elif "if headers:\n\t\t\tprofile_kwargs['headers'] = headers" in content:
        print("✅ Headers check already exists")
    else:
        print("⚠️ Headers check pattern not found")
    
    # 변경 2: CDPClient에 header 추가
    old2 = "self._cdp_client_root = CDPClient(self.cdp_url)"
    new2 = "self._cdp_client_root = CDPClient(self.cdp_url,  additional_headers=self.browser_profile.headers)"
    
    if old2 in content:
        content = content.replace(old2, new2)
        print("✅ Added headers to CDPClient")
    elif "additional_headers=self.browser_profile.headers" in content:
        print("✅ CDPClient headers already exists")
    else:
        print("⚠️ CDPClient pattern not found")
    
    # 파일에 다시 쓰기
    with open(file_path, 'w') as f:
        f.write(content)
    
    print("🎉 Patching complete!")
    return True

if __name__ == "__main__":
    success = patch_browser_use()
    sys.exit(0 if success else 1)

In [ ]:
# browser-use 패치를 적용하는 Python 스크립트 실행
!python patch_browser_use.py

In [ ]:
# Kernel 다시 시작
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

## Live View로 Browser-Use와 Bedrock AgentCore Browser tool 함께 사용하기

여기서는 helper class인 `BrowserViewerServer`를 사용해 Amazon DCV SDK를 통해 Bedrock AgentCore Browser tool에 연결합니다.




In [ ]:
%%writefile live_view_with_browser_use.py
from browser_use import Agent
# from browser_use.browser.session import BrowserSession
from browser_use import Browser, BrowserProfile
from bedrock_agentcore.tools.browser_client import BrowserClient
# from browser_use.browser import BrowserProfile
# from langchain_aws import ChatBedrockConverse
from browser_use.llm import ChatAnthropicBedrock, ChatAWSBedrock
from rich.console import Console
from rich.panel import Panel
from contextlib import suppress
import argparse
import sys
sys.path.append("../interactive_tools")
from browser_viewer import BrowserViewerServer
import asyncio
from boto3.session import Session

console = Console()

boto_session = Session()
region = boto_session.region_name


async def run_browser_task(
    browser_session: Browser, bedrock_chat: ChatAnthropicBedrock, task: str
) -> None:
    """
    browser_use를 사용해 브라우저 자동화 작업을 실행합니다.

    매개변수:
        browser_session: 재사용할 기존 브라우저 세션
        bedrock_chat: Bedrock 채팅 모델 인스턴스
        task: 에이전트가 수행할 자연어 작업
    """
    try:
        # 작업 실행 내용 표시
        console.print(f"\n[bold blue]🤖 Executing task:[/bold blue] {task}")

        # Agent 생성 및 실행
        agent = Agent(task=task, llm=bedrock_chat, browser_session=browser_session)

        # 진행 상태 표시와 함께 실행
        with console.status(
            "[bold green]Running browser automation...[/bold green]", spinner="dots"
        ):
            await agent.run()

        console.print("[bold green]✅ Task completed successfully![/bold green]")

    except Exception as e:
        console.print(f"[bold red]❌ Error during task execution:[/bold red] {str(e)}")
        import traceback

        if console.is_terminal:
            traceback.print_exc()


async def live_view_with_browser_use(prompt, region="us-west-2"):
    """
    에이전트 자동화와 함께 라이브 브라우저 보기를 보여 주는 기본 함수입니다.

    워크플로:
    1. us-west-2 리전에 Amazon Bedrock AgentCore 브라우저 클라이언트를 생성합니다.
    2. 브라우저 초기화를 기다립니다(필수 대기 시간 10초).
    3. 브라우저 제어 기능이 포함된 DCV 기반 라이브 뷰어 서버를 포트 8000에서 시작합니다.
    4. 여러 디스플레이 크기 옵션(720p~1440p)을 구성합니다.
    5. CDP WebSocket을 통해 AI 에이전트 자동화용 브라우저 세션을 설정합니다.
    6. Claude 3.7 Sonnet 모델을 사용해 AI 기반 작업을 실행합니다.
    7. 모든 세션을 올바르게 닫고 브라우저 클라이언트를 중지합니다.

    기능:
    - 웹 인터페이스를 통한 실시간 브라우저 보기
    - 수동 제어권 획득 및 해제 기능
    - browser-use 라이브러리를 활용한 AI 자동화
    - 구성 가능한 디스플레이 레이아웃 및 크기
    """
    console.print(
        Panel(
            "[bold cyan]Browser Live Viewer[/bold cyan]\n\n"
            "This demonstrates:\n"
            "• Live browser viewing with DCV\n"
            "• Configurable display sizes (not limited to 900×800)\n"
            "• Proper display layout callbacks\n\n"
            "[yellow]Note: Requires Amazon DCV SDK files[/yellow]",
            title="Browser Live Viewer",
            border_style="blue",
        )
    )

    try:
        # 1단계: 브라우저 세션 생성
        client = BrowserClient(region)
        client.start()
        
        ws_url, headers = client.generate_ws_headers()

        # 2단계: Viewer server 시작
        console.print("\n[cyan]Step 3: Starting viewer server...[/cyan]")
        viewer = BrowserViewerServer(client, port=8000)
        viewer_url = viewer.start(open_browser=True)

        # 3단계: 기능 표시
        console.print("\n[bold green]Viewer Features:[/bold green]")
        console.print(
            "• Default display: 1600×900 (configured via displayLayout callback)"
        )
        console.print("• Size options: 720p, 900p, 1080p, 1440p")
        console.print("• Real-time display updates")
        console.print("• Take/Release control functionality")

        console.print("\n[yellow]Press Ctrl+C to stop[/yellow]")

        # 4단계: browser-use를 사용해 브라우저와 상호 작용
        # 지속형 브라우저 세션과 모델 생성
        browser_session = None
        bedrock_chat = None

        try:
            # header를 포함한 브라우저 profile 생성
            browser_profile = BrowserProfile(
                headers=headers,
                timeout=1500000,  # 150초 timeout
            )

            # 지속성을 위해 CDP URL과 keep_alive=True를 사용해 브라우저 세션 생성
            browser_session = Browser(
                cdp_url=ws_url,
                browser_profile=browser_profile,
                keep_alive=True,  # 작업 사이에도 브라우저 유지
            )

            # 브라우저 세션 초기화
            console.print("[cyan]🔄 Initializing browser session...[/cyan]")
            await browser_session.start()

            # ChatBedrockConverse를 한 번만 생성
            bedrock_chat = ChatAnthropicBedrock(
                model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
                aws_region=region,
            )

            console.print(
                "[green]✅ Browser session initialized and ready for tasks[/green]\n"
            )

            task = prompt

            await run_browser_task(browser_session, bedrock_chat, task)

        finally:
            # 브라우저 세션 종료
            if browser_session:
                console.print("\n[yellow]🔌 Closing browser session...[/yellow]")
                with suppress(Exception):
                    await browser_session.close()
                console.print("[green]✅ Browser session closed[/green]")
   
    except Exception as e:
        console.print(f"\n[red]Error: {e}[/red]")
        import traceback
        traceback.print_exc()
    finally:
        console.print("\n\n[yellow]Shutting down...[/yellow]")
        if "client" in locals():
            client.stop()
            console.print("✅ Browser session terminated")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True, help="Browser Search instruction")
    parser.add_argument("--region", default="us-west-2", help="AWS region")
    args = parser.parse_args()

    asyncio.run(live_view_with_browser_use(
        args.prompt, args.region
    ))

#### 스크립트 실행하기
아래 스크립트를 실행합니다. 실행 시간은 prompt의 복잡도에 따라 달라질 수 있습니다. 실행이 끝나면 출력을 스크롤해 수행된 액션의 결과를 확인하세요.

In [ ]:
!python live_view_with_browser_use.py --prompt "Search for macbooks on amazon.com and extract the details of the first one" 

## 내부에서는 어떤 일이 일어났을까요? 
* Browser client를 인스턴스화하고 세션을 시작했습니다.
* 그런 다음 `BrowserViewerServer`를 사용해 브라우저 세션에 연결하고 로컬에서 세션 화면을 확인했습니다.
* Browser-Use Agent를 생성하고 브라우저 세션 정보를 전달했습니다.
* 이어서 Browser-Use Agent에 자연어 지시를 보내고 수행되는 액션을 실시간으로 확인했습니다.


# 축하합니다!